# Giai đoạn 3: Củng cố kết quả encoder dùng chung (đường cong dữ liệu, vùng não, độ ổn định, ablation)

## Kết quả giai đoạn 2 (15 subject LOSO, calib 20% = 4,1 phút)
| So sánh | Chênh lệch | Thắng | p |
|---|---|---|---|
| `enc_ridge` − `base` (baseline tuyến tính mạnh nhất) | **+0,0275** (0,206 so với 0,179) | 15/15 | 0,0001 |
| `enc_ridge` − `enc0_ridge` (cùng encoder, chưa train) | +0,0227 | 15/15 | 0,0001 |
| `enc_ridge` − `lin_ridge` (filter bank ngẫu nhiên) | +0,0406 | 15/15 | 0,0001 |
| Như trên nhưng đo bằng `r_no50` (bỏ 46–54 Hz) | +0,0277 | 15/15 | 0,0001 |

- R² của `enc_ridge_cal` là 0,045, so với 0,009 của `base`.
- Với 2 phút calib, encoder (0,182) đã vượt `base` dùng 4 phút (0,179).

## Giai đoạn 3 cần trả lời
| Phần | Câu hỏi | Chi phí |
|---|---|---|
| **3A** Đường cong dữ liệu | Baseline cần bao nhiêu phút calib để bằng encoder? (hình chính của bài) | Không train lại, khoảng 1 giờ |
| **3B** Vùng não / độ sâu | Encoder giúp ở đâu: vỏ não hay cấu trúc sâu? | Vài phút, chỉ đọc kết quả có sẵn |
| **3C** Seed và ablation | Kết quả có ổn định không? Thành phần nào quan trọng (độ dài bộ lọc, envelope, số bước train, độ rộng)? | Mỗi cấu hình khoảng 3 giờ GPU |
| **3D** So sánh các lần chạy | Bảng tổng hợp cho bài báo | Vài giây |
| **3E** So với LeadFormer | Encoder dùng chung có thắng model deep end-to-end trên cùng split không? | Vài giây (nếu `runs/bip_car` đã chạy) |
| **3F** Single-trial | Mức tăng có còn khi bỏ phần ERP (`r_induced`) không? | Không train lại, khoảng 1 giờ |

**Runtime: GPU.** 3A và 3B không cần train lại, nhưng dùng GPU để tính đặc trưng sẽ nhanh hơn.

In [ ]:
#@title 1. Mount Drive + đường dẫn
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, torch
ROOT     = '/content/drive/MyDrive/iEEG_S2I'   #@param {type:"string"}
CODE_DIR = f'{ROOT}/code'
DATA_DIR = f'{ROOT}/data/s2i_v1'
RUNS_DIR = f'{ROOT}/runs'
MAIN_RUN = f'{RUNS_DIR}/shared_bip_car'          # lần chạy chính của giai đoạn 2
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR
os.chdir(CODE_DIR)
print('encoder đã lưu:', len(glob.glob(f'{MAIN_RUN}/enc_sub-*.pt')), '| fold:', len(glob.glob(f'{MAIN_RUN}/fold_sub-*.json')))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ')

## 2. Ghi code lên Drive
- `nonlin.py`: không đổi.
- `shared.py`: thêm cờ `--fast_eval`, chỉ tính các model cần cho seed/ablation, nên nhanh hơn khoảng 2 phút mỗi fold.
- `curve.py`: mới, dùng cho đường cong dữ liệu.

In [ ]:
%%writefile {CODE_DIR}/s2i/nonlin.py
"""
Phase 1c (CPU) - can ANY non-linear EEG feature beat a well-regularised linear model with a
few minutes of calibration?

Phase 1b: the only linear lever that helped was better regularisation (ridge with one alpha
shared by all contacts, chosen by K-fold: +0.008 at 1-2 min, 15/15 subjects). Envelope
features raised alpha-band r by 17 % but the residual stage was selected by Pearson r, which
does not penalise amplitude, so it added noise (R^2 -> -0.16) and hurt small subjects.

Here the second stage is fitted on the OUT-OF-FOLD residual of the linear model and is
  * selected by MSE (not r), per contact, and
  * GATED: switched on for a contact only if its cross-validated MSE beats predicting the
    residual mean.
so it can only add what cross-validation says is real.

Models
------
  ridge         baseline of train_loso.py / ceiling.py (per-contact alpha, one 80/20 split)
  base          ridge_kfold_shared from Phase 1b (new strong baseline)
  env_gated     base + gated stage 2 on alpha/beta band waveforms + theta/alpha/beta envelopes
  wav_gated     base + gated stage 2 on per-channel Morlet envelopes, 12 log-spaced bands 2-64 Hz
                (exhaustive per-channel spectral power, incl. gamma)
  rocket_gated  base + gated stage 2 on random non-linear convolutional features
                (random spatial filter -> random Morlet wavelet -> envelope or ReLU).
                A cheap stand-in for "a non-linear encoder", with no training.
  all_gated     base + gated stage 2 on [wavelet envelopes, random features]

Usage
-----
python -m s2i.nonlin --data DATA_DIR --out RUNS_DIR/nonlin_bip_car
"""
import argparse
import glob
import json
import os
import sys
import time

import numpy as np
from scipy.fft import next_fast_len
from scipy.signal import butter, hilbert, sosfiltfilt

from s2i.baselines import lagged
from s2i.ceiling import StreamRidge, chrono_split
from s2i.metrics import per_contact_metrics
from s2i.within import ALPHAS_WIDE, LAG9, KFoldRidge, _blocks

FS = 200
ENV_LAGS = (-20, -10, 0, 10, 20)
_B = {k: butter(4, f, btype="band", fs=FS, output="sos")
      for k, f in {"theta": (4, 8), "alpha": (8, 13), "beta": (13, 30)}.items()}


# ----------------------------------------------------------------------------- features
def feat_env(e):
    a = sosfiltfilt(_B["alpha"], e, axis=1)
    b = sosfiltfilt(_B["beta"], e, axis=1)
    t = sosfiltfilt(_B["theta"], e, axis=1)
    parts = [a, b, np.abs(hilbert(t, axis=1)), np.abs(hilbert(a, axis=1)), np.abs(hilbert(b, axis=1))]
    return lagged(np.concatenate(parts, 0).astype(np.float32), ENV_LAGS)


def morlet(freq, cycles):
    """Complex Morlet kernels (len(freq), 2H+1), unit L1 norm, centred at index H."""
    sig = np.asarray(cycles) / (2 * np.pi * np.asarray(freq))        # Gaussian width, s
    H = int(np.ceil(3 * sig.max() * FS))
    t = np.arange(-H, H + 1) / FS
    w = np.exp(-t[None] ** 2 / (2 * sig[:, None] ** 2)) * np.exp(2j * np.pi * np.asarray(freq)[:, None] * t[None])
    return (w / np.abs(w).sum(1, keepdims=True)).astype(np.complex64), H


def wconv(S, kern, H):
    """Zero-phase convolution of each row of S (M, L) with its own kernel (M, 2H+1) -> complex (M, L)."""
    L = S.shape[1]
    n = next_fast_len(L + 2 * H)
    return np.fft.ifft(np.fft.fft(S, n) * np.fft.fft(kern, n), n)[:, H:H + L]


class MorletEnvBank:
    """Per-channel Morlet envelope at n_freq log-spaced frequencies (all channels x all bands), lagged."""

    def __init__(self, freqs=tuple(np.geomspace(2, 64, 12)), cycles=5, lags=(-20, 0, 20)):
        self.kern1, self.H = morlet(freqs, np.full(len(freqs), cycles))
        self.nf, self.lags = len(freqs), lags

    def __call__(self, e):
        C = e.shape[0]
        S = np.repeat(e, self.nf, 0)
        env = np.abs(wconv(S, np.tile(self.kern1, (C, 1)), self.H))
        return lagged(env.astype(np.float32), self.lags)


class RandomConvFeatures:
    """
    M random (spatial filter -> complex Morlet wavelet -> non-linearity) features, fixed seed.
    A cheap stand-in for the first layer of a convolutional encoder, with no training:
      spatial filter : 1/3 single channel, 1/3 channel-pair difference, 1/3 dense random
      wavelet        : centre frequency log-uniform in [fmin, fmax] Hz, 3-7 cycles
      non-linearity  : half envelope |w * s|, half ReLU(Re(w * s) - b * rms), b ~ U(-1, 1)
    mode (same random filters/wavelets whatever the mode, so modes form a matched ablation):
      "mix" (above) | "env" all envelopes | "relu" all ReLU | "linear" Re(w * s), no non-linearity
    Synthetic check: dense spatial filters + short random 9-tap kernels (plain ROCKET) missed an
    alpha-envelope signal that feat_env recovered - broadband kernels dilute narrow-band power.
    """

    def __init__(self, n_chan, M=512, fmin=2.0, fmax=60.0, seed=0, mode="mix"):
        assert mode in ("mix", "env", "relu", "linear")
        self.mode = mode
        g = np.random.default_rng(seed)
        self.Ws = np.zeros((M, n_chan), np.float32)
        kind = np.arange(M) % 3
        for i in range(M):
            if kind[i] == 0 or n_chan < 2:
                self.Ws[i, g.integers(n_chan)] = 1
            elif kind[i] == 1:
                c = g.choice(n_chan, 2, replace=False)
                self.Ws[i, c] = (1, -1)
            else:
                self.Ws[i] = g.normal(size=n_chan)
        self.Ws /= np.linalg.norm(self.Ws, axis=1, keepdims=True)
        self.freq = np.exp(g.uniform(np.log(fmin), np.log(fmax), M))
        self.kern, self.H = morlet(self.freq, g.uniform(3, 7, M))
        self.env = g.random(M) < 0.5
        self.bias = g.uniform(-1, 1, M).astype(np.float32)

    def __call__(self, e):
        C = wconv(self.Ws @ e, self.kern, self.H)
        R = C.real
        if self.mode == "linear":
            return R.astype(np.float32)
        if self.mode == "env":
            return np.abs(C).astype(np.float32)
        out = np.maximum(R - self.bias[:, None] * np.sqrt((R * R).mean(1, keepdims=True)), 0)
        if self.mode == "mix":
            out[self.env] = np.abs(C[self.env])
        return out.astype(np.float32)


def concat_feats(*fs):
    return lambda e: np.concatenate([f(e) for f in fs], 0)


# ----------------------------------------------------------------------------- gated stage 2
class GatedResidualRidge:
    """
    Ridge on a residual target with per-contact alpha chosen by K-fold MSE, and a per-contact
    gate: ON only if the best CV MSE beats predicting the training-fold mean by `margin`.
    """

    def __init__(self, feat, alphas=ALPHAS_WIDE, k=5, margin=0.002):
        self.feat, self.alphas, self.k, self.margin = feat, alphas, k, margin
        self._sr = StreamRidge(feat, alphas)

    def fit(self, E, R):
        n, K = len(E), R[0].shape[0]
        folds = _blocks(n, self.k)
        fold_of = np.empty(n, int)
        for fi, idx in enumerate(folds):
            fold_of[idx] = fi
        st = [self._sr._stats([E[i] for i in idx], [R[i] for i in idx]) for idx in folds]
        total = [sum(x) for x in zip(*st)]
        models = [StreamRidge._solve([a - b for a, b in zip(total, s)]) for s in st]
        sse = np.zeros((len(self.alphas), K))
        sse_off = np.zeros(K)
        for i in range(n):
            s = models[fold_of[i]]
            t = R[i].T.astype(np.float64)
            Z = self._sr._design(s, E[i]) @ s["Q"]
            for ai, a in enumerate(self.alphas):
                p = Z @ (s["QB"] / (s["lam"] + a * s["scale"])[:, None]) + s["ym"]
                sse[ai] += ((p - t) ** 2).sum(0)
            sse_off += ((s["ym"] - t) ** 2).sum(0)
        best = np.argmin(sse, 0)
        self.gate = sse[best, np.arange(K)] < sse_off * (1 - self.margin)
        full = StreamRidge._solve(total)
        W = np.zeros((len(full["mu"]), K))
        for ai, a in enumerate(self.alphas):
            cols = np.where((best == ai) & self.gate)[0]
            if len(cols):
                W[:, cols] = full["Q"] @ (full["QB"][:, cols] / (full["lam"] + a * full["scale"])[:, None])
        self.s, self.W = full, W
        return self

    def predict(self, E):
        if not self.gate.any():
            return [np.zeros((len(self.gate), e.shape[1]), np.float32) for e in E]
        return [(self._sr._design(self.s, e) @ self.W + self.s["ym"] * self.gate).T.astype(np.float32) for e in E]


def gated(base, Ec, Yc, Et, feat):
    m = GatedResidualRidge(feat).fit(Ec, [y - o for y, o in zip(Yc, base.oof)])
    return [p + q for p, q in zip(base.predict(Et), m.predict(Et))], float(m.gate.mean())


# ----------------------------------------------------------------------------- per subject
def analyze(E, Y, a, log=print):
    n = len(E)
    _, test = chrono_split(n, 0.2, gap=1)
    Et, Yt = [E[i] for i in test], [Y[i] for i in test]
    rfeat = RandomConvFeatures(E[0].shape[0], M=a.n_random, seed=a.seed)
    wfeat = MorletEnvBank()
    res = {"n_trials": n, "fracs": {}}
    for f in a.fracs:
        n_c = max(3, int(round(n * f)))
        if n_c >= test[0]:
            continue
        Ec, Yc = E[:n_c], Y[:n_c]
        bands = abs(f - 0.2) < 1e-9
        out, on = {}, {}

        def score(name, P):
            m = per_contact_metrics(P, Yt, FS, bands=bands)
            out[name] = {"mean": {k: float(np.nanmean(v)) for k, v in m.items()}, "r": [float(x) for x in m["r"]]}

        score("ridge", StreamRidge(LAG9).fit(Ec, Yc).predict(Et))
        base = KFoldRidge(LAG9, shared=True).fit(Ec, Yc)
        score("base", base.predict(Et))
        P, on["env_gated"] = gated(base, Ec, Yc, Et, feat_env)
        score("env_gated", P)
        P, on["wav_gated"] = gated(base, Ec, Yc, Et, wfeat)
        score("wav_gated", P)
        P, on["rocket_gated"] = gated(base, Ec, Yc, Et, rfeat)
        score("rocket_gated", P)
        if not a.no_all:
            P, on["all_gated"] = gated(base, Ec, Yc, Et, concat_feats(wfeat, rfeat))
            score("all_gated", P)
        mins = sum(e.shape[1] for e in Ec) / FS / 60
        res["fracs"][f"{f:g}"] = {"n_calib": n_c, "minutes": mins, "gate_on": on,
                                  "mean": {k: v["mean"]["r"] for k, v in out.items()},
                                  "bands": {k: v["mean"] for k, v in out.items()} if bands else {},
                                  "r": {k: v["r"] for k, v in out.items()}}
        log(f"    f={f:g} ({mins:.1f} min) " + " ".join(f"{k}={v['mean']['r']:+.3f}" for k, v in out.items()) +
            " | gate on: " + " ".join(f"{k}={v:.2f}" for k, v in on.items()))
    return res


def summarize(results, out_dir, log=print):
    from scipy.stats import wilcoxon

    def wp(x, y):
        try:
            return float(wilcoxon(x, y).pvalue)
        except ValueError:
            return float("nan")

    lines, summ = [], {}
    fracs = sorted({f for r in results for f in r["fracs"]}, key=float)
    for f in fracs:
        rows = [r for r in results if f in r["fracs"]]
        models = list(rows[0]["fracs"][f]["mean"])
        M = {m: np.array([r["fracs"][f]["mean"][m] for r in rows]) for m in models}
        mins = np.mean([r["fracs"][f]["minutes"] for r in rows])
        lines += [f"\n### calib fraction {f}  ({len(rows)} subjects, mean {mins:.1f} min)",
                  "| model | mean r | vs ridge | p | vs base | wins vs base | p | gate on |",
                  "|---|---|---|---|---|---|---|---|"]
        summ[f] = {}
        for m in models:
            d0, d1 = M[m] - M["ridge"], M[m] - M["base"]
            gon = np.mean([r["fracs"][f]["gate_on"].get(m, np.nan) for r in rows])
            p0 = wp(M[m], M["ridge"]) if m != "ridge" else float("nan")
            p1 = wp(M[m], M["base"]) if m not in ("ridge", "base") else float("nan")
            summ[f][m] = {"mean": float(M[m].mean()), "diff_ridge": float(d0.mean()), "p_ridge": p0,
                          "diff_base": float(d1.mean()), "wins_base": int((d1 > 0).sum()), "p_base": p1,
                          "gate_on": float(gon)}
            lines.append(f"| {m} | {M[m].mean():.3f} | {d0.mean():+.4f} | {p0:.4f} | {d1.mean():+.4f} | "
                         f"{(d1 > 0).sum()}/{len(d1)} | {p1:.4f} | {gon:.2f} |")
    if "0.2" in fracs:
        rows = [r for r in results if "0.2" in r["fracs"]]
        models = list(rows[0]["fracs"]["0.2"]["bands"])
        keys = ["r", "r2", "r_delta", "r_theta", "r_alpha", "r_beta", "env_alpha", "env_beta", "env_gamma"]
        keys += [k for k in ("r_no50", "r_induced") if k in rows[0]["fracs"]["0.2"]["bands"][models[0]]]
        lines += ["\n### calib fraction 0.2: by band", "| model | " + " | ".join(keys) + " |",
                  "|---" * (len(keys) + 1) + "|"]
        for m in models:
            lines.append(f"| {m} | " + " | ".join(
                f"{np.nanmean([r['fracs']['0.2']['bands'][m].get(k, np.nan) for r in rows]):.3f}" for k in keys) + " |")
        models = list(rows[0]["fracs"]["0.2"]["mean"])
        lines += ["\n### per subject (calib fraction 0.2)", "| subject | " + " | ".join(models) + " |",
                  "|---" * (len(models) + 1) + "|"]
        for r in rows:
            lines.append(f"| {r['subject']} | " + " | ".join(f"{r['fracs']['0.2']['mean'][m]:.3f}" for m in models) + " |")
    md = "\n".join(lines)
    log(md)
    if out_dir:
        with open(os.path.join(out_dir, "summary.md"), "w", encoding="utf-8") as fh:
            fh.write(md + "\n")
        with open(os.path.join(out_dir, "summary.json"), "w") as fh:
            json.dump(summ, fh, indent=1)
    return summ


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--min_trials", type=int, default=20)
    ap.add_argument("--fracs", type=float, nargs="*", default=[0.05, 0.1, 0.2])
    ap.add_argument("--n_random", type=int, default=512)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--no_all", action="store_true")
    ap.add_argument("--overwrite", action="store_true")
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    os.makedirs(a.out, exist_ok=True)
    log_f = open(os.path.join(a.out, "nonlin.log"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} | {msg}"
        print(line, flush=True)
        log_f.write(line + "\n")
        log_f.flush()

    from s2i.ceiling import load_trials
    log(f"config: {vars(a)}")
    results = []
    for p in sorted(glob.glob(os.path.join(a.data, "sub-[0-9][0-9].pt"))):
        sid = os.path.basename(p)[:-3]
        fp = os.path.join(a.out, f"nonlin_{sid}.json")
        if os.path.exists(fp) and not a.overwrite:
            results.append(json.load(open(fp)))
            log(f"[{sid}] already done, loaded")
            continue
        t0 = time.time()
        s, E, Y = load_trials(p, "bip", "car")
        if len(E) < a.min_trials or s.n_targets == 0:
            log(f"[{sid}] skipped")
            continue
        log(f"[{sid}] {len(E)} trials, {len(s.eeg_names)} scalp, {s.n_targets} targets")
        r = analyze(E, Y, a, log)
        r["subject"] = sid
        with open(fp + ".tmp", "w") as fh:
            json.dump(r, fh)
        os.replace(fp + ".tmp", fp)
        results.append(r)
        log(f"[{sid}] done in {time.time() - t0:.0f}s")
    if results:
        summarize(results, a.out, log)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/shared.py
"""
Phase 2 - shared temporal encoder + per-subject ridge decoder (LOSO, GPU).

Evidence from phases 0-1d (15 subjects, bipolar iEEG, CAR EEG, 20 % chronological calibration):
  * the SPATIAL scalp -> contact mapping does not transfer across subjects (1a: zero-shot r ~ 0);
  * within a subject the lever is frequency-selective, long-support LINEAR temporal filtering
    (1d: random Morlet filter bank, linear, gated on top of the ridge: +0.0075, 14/15 subjects),
    while non-linearity and more random features add ~nothing at 4 min of calibration.
Hypothesis: the TEMPORAL part (which filters of the scalp signal predict intracranial activity)
is shared across subjects even though the spatial part is not. Hence:
  encoder : per-channel filter bank, shared by all channels and all subjects, trained on the
            OTHER subjects. Quadrature pairs initialised as complex Morlet 2-60 Hz (+-1.2 s
            support) + the envelope of each pair + a few free filters.
  decoder : per-subject linear readout (channels x filters -> contacts), fitted in closed form by
            ridge on the calibration trials only, exactly like the baselines.
Training is episodic so it matches deployment: each step takes a training subject, fits a ridge
readout on ~4 min of consecutive trials (support), scores Pearson r on the following trials
(query), and back-propagates through the ridge solve into the filters. Model selection: the
same procedure on 2 held-out validation subjects (first 20 % of trials -> later trials).

Evaluation on the test subject (never seen in training; same split and calibration sizes as
phases 0-1d, so ridge / base / lin_gated must reproduce the phase-1d numbers):
  ridge, base, lin_gated  baselines (lin_gated = best model of phase 1d: the bar to beat)
  enc0_gated              base + gated stage 2 on the encoder AT INITIALISATION (untrained Morlet
                          bank): control - does the gain come from learning or from the init?
  lin_ridge, enc0_ridge   single-stage controls: ridge (K-fold, shared alpha) on the random linear
                          Morlet bank / on the UNTRAINED encoder
  enc_ridge               ridge (K-fold, shared alpha) on learned encoder features alone
  enc_ridge_cal           enc_ridge with a per-contact amplitude gain fitted on out-of-fold
                          calibration predictions (same r, honest R^2)
Every model is also scored with
  r_no50     46-54 Hz removed from prediction and target (the trained filters develop a sharp 50 Hz
             peak: mains residue after the notch filter);
  r_induced  each side minus its own trial average over the test trials, i.e. how well
             the SINGLE-TRIAL (non stimulus-locked) part is predicted.
  enc_gated               base + gated stage 2 on learned encoder features      <- main model

Usage
-----
python -m s2i.shared --data DATA_DIR --out RUNS_DIR/shared_bip_car
"""
import argparse
import copy
import glob
import json
import os
import sys
import time

import numpy as np
import torch
import torch.nn as nn
from scipy.signal import butter, sosfiltfilt

from s2i.ceiling import StreamRidge, chrono_split, load_trials, template, tmpl_at
from s2i.metrics import per_contact_metrics
from s2i.nonlin import RandomConvFeatures, gated, summarize
from s2i.within import ALPHAS_WIDE, LAG9, KFoldRidge

FS = 200
EXTRA_METRICS = ("r_no50", "r_induced")


# ----------------------------------------------------------------------------- encoder
class TemporalEncoder(nn.Module):
    """Per-channel filter bank shared by all channels and subjects: (B, C, L) -> (B, C, F, L)."""

    def __init__(self, n_pairs=12, half=240, fmin=2.0, fmax=60.0, cycles=5.0, n_free=4, envelope=True):
        super().__init__()
        self.cfg = dict(n_pairs=n_pairs, half=half, fmin=fmin, fmax=fmax, cycles=cycles, n_free=n_free,
                        envelope=envelope)
        self.n_pairs, self.envelope = n_pairs, envelope
        t = np.arange(-half, half + 1) / FS
        f = np.geomspace(fmin, fmax, n_pairs)
        gauss = np.exp(-t[None] ** 2 / (2 * (cycles / (2 * np.pi * f[:, None])) ** 2))
        w = np.zeros((2 * n_pairs + n_free, t.size))
        w[0:2 * n_pairs:2] = gauss * np.cos(2 * np.pi * f[:, None] * t[None])
        w[1:2 * n_pairs:2] = gauss * np.sin(2 * np.pi * f[:, None] * t[None])
        if n_free:
            w[2 * n_pairs, half] = 1.0                                   # identity (broadband, lag 0)
            g = np.random.default_rng(0)
            w[2 * n_pairs + 1:] = g.normal(size=(n_free - 1, t.size)) * np.exp(-t ** 2 / (2 * 0.05 ** 2))
        w /= np.linalg.norm(w, axis=1, keepdims=True)
        self.conv = nn.Conv1d(1, w.shape[0], 2 * half + 1, padding=half, bias=False)
        with torch.no_grad():
            self.conv.weight.copy_(torch.tensor(w, dtype=torch.float32)[:, None])

    @property
    def n_feat(self):
        return self.conv.out_channels + (self.n_pairs if self.envelope else 0)

    def forward(self, x):
        B, C, L = x.shape
        h = self.conv(x.reshape(B * C, 1, L))
        if self.envelope:
            p = h[:, :2 * self.n_pairs].float()
            h = torch.cat([h.float(), torch.sqrt(p[:, 0::2] ** 2 + p[:, 1::2] ** 2 + 1e-6)], 1)
        return h.reshape(B, C, -1, L)


# ----------------------------------------------------------------------------- torch helpers
def pad_trials(E, Y, idx, dev):
    L = max(E[i].shape[1] for i in idx)
    x = np.zeros((len(idx), E[idx[0]].shape[0], L), np.float32)
    y = np.zeros((len(idx), Y[idx[0]].shape[0], L), np.float32)
    m = np.zeros((len(idx), L), bool)
    for j, i in enumerate(idx):
        n = E[i].shape[1]
        x[j, :, :n], y[j, :, :n], m[j, :n] = E[i], Y[i], True
    return torch.from_numpy(x).to(dev), torch.from_numpy(y).to(dev), torch.from_numpy(m).to(dev)


def rows(enc, E, Y, idx, dev, amp, chunk=16):
    """Encoder features and targets of the given trials as (N, C*F) / (N, K) rows (padding removed)."""
    X, T = [], []
    for s in range(0, len(idx), chunk):
        x, y, m = pad_trials(E, Y, idx[s:s + chunk], dev)
        with torch.autocast(device_type=dev.type, enabled=amp):
            h = enc(x)
        B, C, F, L = h.shape
        X.append(h.float().permute(0, 3, 1, 2).reshape(B, L, C * F)[m])
        T.append(y.permute(0, 2, 1)[m])
    return torch.cat(X), torch.cat(T)


def ridge_torch(Xs, Ys, Xq, lam):
    """Ridge on standardised features; lam is relative to the mean eigenvalue (= 1 after scaling)."""
    mu, sd = Xs.mean(0), Xs.std(0) + 1e-4
    A, Q = (Xs - mu) / sd, (Xq - mu) / sd
    ym = Ys.mean(0)
    N, D = A.shape
    G = A.T @ A / N + lam * torch.eye(D, device=A.device)
    return Q @ torch.linalg.solve(G, A.T @ (Ys - ym) / N) + ym


def col_r(P, T):
    P, T = P - P.mean(0), T - T.mean(0)
    return (P * T).sum(0) / ((P * P).sum(0).sqrt() * (T * T).sum(0).sqrt() + 1e-8)


def episode(n_trials, mean_len, rng, sup_sec, n_query):
    """Chronological episode: ~sup_sec of consecutive support trials, gap 1, then n_query trials."""
    n_q = min(n_query, max(1, n_trials // 4))
    n_s = int(np.clip(round(sup_sec * FS / mean_len), 3, n_trials - n_q - 1))
    start = int(rng.integers(0, n_trials - n_s - n_q - 1 + 1))
    return list(range(start, start + n_s)), list(range(start + n_s + 1, start + n_s + 1 + n_q))


@torch.no_grad()
def val_score(enc, lam, subjects, dev, amp, max_test):
    enc.eval()
    rs = []
    for _, E, Y in subjects:
        cal, test = chrono_split(len(E), 0.2, gap=1)
        Xs, Ys = rows(enc, E, Y, cal, dev, amp)
        Xq, Yq = rows(enc, E, Y, test[:max_test], dev, amp)
        rs.append(col_r(ridge_torch(Xs, Ys, Xq, lam), Yq).mean().item())
    return float(np.mean(rs))


# ----------------------------------------------------------------------------- training
def train_encoder(a, train, val, dev, log, ckpt_path):
    enc = TemporalEncoder(**enc_cfg(a)).to(dev)
    log_lam = nn.Parameter(torch.tensor(float(np.log(a.lam0)), device=dev))
    opt = torch.optim.AdamW([{"params": enc.parameters(), "lr": a.lr, "weight_decay": 0.0},
                             {"params": [log_lam], "lr": a.lr_lam, "weight_decay": 0.0}])
    amp = a.amp and dev.type == "cuda"
    rng = np.random.default_rng(a.seed)
    mean_len = {sid: float(np.mean([e.shape[1] for e in E])) for sid, E, _ in train}
    step, best, best_state, bad, hist = 0, -1e9, None, 0, []
    if ckpt_path and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=dev, weights_only=False)
        enc.load_state_dict(ck["enc"])
        opt.load_state_dict(ck["opt"])
        with torch.no_grad():
            log_lam.copy_(ck["log_lam"])
        step, best, best_state, bad, hist = ck["step"], ck["best"], ck["best_state"], ck["bad"], ck["hist"]
        rng = np.random.default_rng(a.seed + step)
        log(f"  resumed at step {step} (best val r={best:.4f})")
    else:
        r0 = val_score(enc, log_lam.exp().item(), val, dev, amp, a.val_trials)
        hist.append((0, r0))
        log(f"  step     0 | val r (Morlet init) = {r0:.4f}")
        best, best_state = r0, (copy.deepcopy(enc.state_dict()), log_lam.exp().item())
    t0, run_r = time.time(), []
    while step < a.steps and bad < a.patience:
        enc.train()
        loss = 0.0
        for si in rng.choice(len(train), a.subj_per_step, replace=len(train) < a.subj_per_step):
            sid, E, Y = train[si]
            sup, qry = episode(len(E), mean_len[sid], rng, a.sup_sec, a.n_query)
            Xs, Ys = rows(enc, E, Y, sup, dev, amp)
            Xq, Yq = rows(enc, E, Y, qry, dev, amp)
            r = col_r(ridge_torch(Xs, Ys, Xq, log_lam.exp()), Yq).mean()
            loss = loss + (1 - r) / a.subj_per_step
            run_r.append(r.item())
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
        opt.step()
        step += 1
        if step % a.eval_every == 0 or step == a.steps:
            vr = val_score(enc, log_lam.exp().item(), val, dev, amp, a.val_trials)
            hist.append((step, vr))
            if vr > best:
                best, bad, best_state = vr, 0, (copy.deepcopy(enc.state_dict()), log_lam.exp().item())
            else:
                bad += 1
            log(f"  step {step:5d} | train query r={np.mean(run_r):.4f} | val r={vr:.4f} | "
                f"lam={log_lam.exp().item():.3g} | {time.time() - t0:.0f}s" + (" *" if bad == 0 else ""))
            run_r, t0 = [], time.time()
            if ckpt_path:
                torch.save({"enc": enc.state_dict(), "opt": opt.state_dict(), "log_lam": log_lam.detach(),
                            "step": step, "best": best, "best_state": best_state, "bad": bad, "hist": hist},
                           ckpt_path + ".tmp")
                os.replace(ckpt_path + ".tmp", ckpt_path)
    enc.load_state_dict(best_state[0])
    return enc, best_state[1], best, hist


def enc_cfg(a):
    return dict(n_pairs=a.n_pairs, half=a.half, fmin=a.fmin, fmax=a.fmax, cycles=a.cycles,
                n_free=a.n_free, envelope=not a.no_env)


# ----------------------------------------------------------------------------- evaluation
def np_feat(enc, dev, amp):
    """Encoder as a numpy feature function (C, L) -> (C*F, L) for the StreamRidge machinery."""
    enc = enc.eval()

    @torch.no_grad()
    def f(e):
        x = torch.from_numpy(np.ascontiguousarray(e, np.float32))[None].to(dev)
        with torch.autocast(device_type=dev.type, enabled=amp):
            h = enc(x)[0]
        return h.float().reshape(-1, h.shape[-1]).cpu().numpy()
    return f


_LINE = butter(4, (46, 54), btype="bandstop", fs=FS, output="sos")


def r_without_line(P, T):
    """Per-contact r after removing 46-54 Hz (mains residue) from BOTH prediction and target."""
    fp = [sosfiltfilt(_LINE, p, axis=1) for p in P]
    ft = [sosfiltfilt(_LINE, t, axis=1) for t in T]
    return per_contact_metrics(fp, ft, FS, bands=False)["r"]


def r_single_trial(P, T):
    """
    Per-contact r of the single-trial parts: each side minus ITS OWN trial average over the test
    trials (prediction minus the mean prediction, target minus the mean target). Removing the same
    template from both sides would add a shared component and inflate r.
    """
    tP, tT = template(P), template(T)
    return per_contact_metrics([p - tmpl_at(tP, p.shape[1]) for p in P],
                               [t - tmpl_at(tT, t.shape[1]) for t in T], FS, bands=False)["r"]


def amplitude_calibrate(kf, Yc, P):
    """
    Per-contact least-squares gain on the OUT-OF-FOLD calibration predictions. Alpha is chosen by
    r, which ignores amplitude; this fixes the amplitude (R^2) without changing r (gain >= 0).
    """
    ym = kf.s["ym"][:, None]
    num = sum(((o - ym) * (y - ym)).sum(1) for o, y in zip(kf.oof, Yc))
    den = sum(((o - ym) ** 2).sum(1) for o in kf.oof)
    g = np.clip(num / np.maximum(den, 1e-12), 0, None)[:, None]
    return [(ym + g * (p - ym)).astype(np.float32) for p in P]


def evaluate(E, Y, enc0, enc, a, dev, log):
    amp = a.amp and dev.type == "cuda"
    n = len(E)
    _, test = chrono_split(n, 0.2, gap=1)
    Et, Yt = [E[i] for i in test], [Y[i] for i in test]
    f0, f1 = np_feat(enc0, dev, amp), np_feat(enc, dev, amp)
    lin = RandomConvFeatures(E[0].shape[0], M=512, seed=0, mode="linear")
    res = {"n_trials": n, "fracs": {}}
    for f in a.fracs:
        n_c = max(3, int(round(n * f)))
        if n_c >= test[0]:
            continue
        Ec, Yc = E[:n_c], Y[:n_c]
        bands = abs(f - 0.2) < 1e-9
        out, on = {}, {}

        def score(name, P):
            m = per_contact_metrics(P, Yt, FS, bands=bands)
            m["r_no50"] = r_without_line(P, Yt)
            m["r_induced"] = r_single_trial(P, Yt)
            out[name] = {"mean": {k: float(np.nanmean(v)) for k, v in m.items()}, "r": [float(x) for x in m["r"]],
                         **{x: [float(v) for v in m[x]] for x in EXTRA_METRICS}}

        score("ridge", StreamRidge(LAG9).fit(Ec, Yc).predict(Et))
        base = KFoldRidge(LAG9, shared=True).fit(Ec, Yc)
        score("base", base.predict(Et))
        if not a.fast_eval:
            for name, feat in [("lin_gated", lin), ("enc0_gated", f0), ("enc_gated", f1)]:
                P, on[name] = gated(base, Ec, Yc, Et, feat)
                score(name, P)
        for name, feat in [("lin_ridge", lin)] * (not a.fast_eval) + [("enc0_ridge", f0)]:
            score(name, KFoldRidge(feat, ALPHAS_WIDE, shared=True, keep_oof=False).fit(Ec, Yc).predict(Et))
        kf = KFoldRidge(f1, ALPHAS_WIDE, shared=True).fit(Ec, Yc)
        P = kf.predict(Et)
        score("enc_ridge", P)
        score("enc_ridge_cal", amplitude_calibrate(kf, Yc, P))
        mins = sum(e.shape[1] for e in Ec) / FS / 60
        res["fracs"][f"{f:g}"] = {"n_calib": n_c, "minutes": mins, "gate_on": on,
                                  "mean": {k: v["mean"]["r"] for k, v in out.items()},
                                  **{x: {k: v[x] for k, v in out.items()} for x in EXTRA_METRICS},
                                  "bands": {k: v["mean"] for k, v in out.items()} if bands else {},
                                  "r": {k: v["r"] for k, v in out.items()}}
        log(f"    f={f:g} ({mins:.1f} min) " + " ".join(f"{k}={v['mean']['r']:+.3f}" for k, v in out.items()) +
            " | gate on: " + " ".join(f"{k}={v:.2f}" for k, v in on.items()))
    return res


KEY_TESTS = [("enc_ridge", "enc0_ridge", "learning helps (vs same encoder untrained)?"),
             ("enc_ridge", "lin_ridge", "learned filters beat random filters?"),
             ("enc0_ridge", "lin_ridge", "Morlet init beats random filters?"),
             ("enc_ridge", "base", "beats the strong linear baseline?"),
             ("lin_ridge", "base", "single-stage filter bank beats LAG9?")]


def key_tests(results, out_dir, log):
    """Paired subject-level tests for the decisive comparisons, on r and on every EXTRA_METRICS."""
    from scipy.stats import wilcoxon
    lines, js = ["\n### key comparisons (paired over subjects)",
                 "| calib | question | A | B | metric | mean A | mean B | A - B | A wins | Wilcoxon p |",
                 "|---|---|---|---|---|---|---|---|---|---|"], []
    fracs = sorted({f for r in results for f in r["fracs"]}, key=float)
    for f in fracs:
        rows = [r for r in results if f in r["fracs"] and "r_no50" in r["fracs"][f]]
        metrics = ("r",) + tuple(x for x in EXTRA_METRICS if rows and x in rows[0]["fracs"][f])
        for ma, mb, q in KEY_TESTS:
            if not rows or ma not in rows[0]["fracs"][f]["mean"] or mb not in rows[0]["fracs"][f]["mean"]:
                continue
            for metric in metrics:
                if metric == "r":
                    get = lambda r, m: r["fracs"][f]["mean"][m]                         # noqa: E731
                else:
                    get = lambda r, m: float(np.nanmean(r["fracs"][f][metric][m]))     # noqa: E731
                x = np.array([get(r, ma) for r in rows])
                y = np.array([get(r, mb) for r in rows])
                try:
                    p = float(wilcoxon(x, y).pvalue)
                except ValueError:
                    p = float("nan")
                js.append({"frac": f, "a": ma, "b": mb, "metric": metric, "mean_a": float(x.mean()),
                           "mean_b": float(y.mean()), "diff": float((x - y).mean()), "wins": int((x > y).sum()),
                           "n": len(x), "p": p})
                lines.append(f"| {f} | {q} | {ma} | {mb} | {metric} | {x.mean():.3f} | {y.mean():.3f} | "
                             f"{(x - y).mean():+.4f} | {(x > y).sum()}/{len(x)} | {p:.4f} |")
    md = "\n".join(lines)
    log(md)
    with open(os.path.join(out_dir, "summary.md"), "a", encoding="utf-8") as fh:
        fh.write(md + "\n")
    with open(os.path.join(out_dir, "key_tests.json"), "w") as fh:
        json.dump(js, fh, indent=1)


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--subjects", nargs="*", default=None, help="subset of subjects to load")
    ap.add_argument("--folds", nargs="*", default=None, help="test subjects to run (default: all)")
    ap.add_argument("--min_trials", type=int, default=20)
    ap.add_argument("--fracs", type=float, nargs="*", default=[0.1, 0.2])
    # encoder
    ap.add_argument("--n_pairs", type=int, default=12)
    ap.add_argument("--half", type=int, default=240, help="filter half-length in samples (240 = 1.2 s)")
    ap.add_argument("--fmin", type=float, default=2.0)
    ap.add_argument("--fmax", type=float, default=60.0)
    ap.add_argument("--cycles", type=float, default=5.0)
    ap.add_argument("--n_free", type=int, default=4)
    ap.add_argument("--no_env", action="store_true")
    # episodic training
    ap.add_argument("--steps", type=int, default=3000)
    ap.add_argument("--eval_every", type=int, default=150)
    ap.add_argument("--patience", type=int, default=6)
    ap.add_argument("--subj_per_step", type=int, default=2)
    ap.add_argument("--sup_sec", type=float, default=240.0, help="support size (seconds), ~4 min as at test")
    ap.add_argument("--n_query", type=int, default=8)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--lr_lam", type=float, default=1e-2)
    ap.add_argument("--lam0", type=float, default=1.0)
    ap.add_argument("--n_val_subj", type=int, default=2)
    ap.add_argument("--val_trials", type=int, default=40, help="test trials per validation subject")
    ap.add_argument("--no_amp", action="store_true")
    ap.add_argument("--fast_eval", action="store_true",
                    help="only ridge, base, enc0_ridge, enc_ridge, enc_ridge_cal (for seed / ablation runs)")
    ap.add_argument("--reeval", action="store_true",
                    help="redo the evaluation of finished folds with their saved encoders (no retraining)")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    a.amp = not a.no_amp
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    os.makedirs(a.out, exist_ok=True)
    log_f = open(os.path.join(a.out, "shared.log"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} | {msg}"
        print(line, flush=True)
        log_f.write(line + "\n")
        log_f.flush()

    dev = torch.device(a.device)
    log(f"config: {vars(a)}")
    subjects = []
    for p in sorted(glob.glob(os.path.join(a.data, "sub-[0-9][0-9].pt"))):
        sid = os.path.basename(p)[:-3]
        if a.subjects and sid not in a.subjects:
            continue
        s, E, Y = load_trials(p, "bip", "car")
        if len(E) < a.min_trials or s.n_targets == 0:
            log(f"skip {sid}: {len(E)} trials, {s.n_targets} targets")
            continue
        subjects.append((sid, E, Y))
        log(f"{sid}: {len(E)} trials, {E[0].shape[0]} scalp, {s.n_targets} targets")
    folds = [s for s in subjects if not a.folds or s[0] in a.folds]

    results = []
    for fi, (sid, E, Y) in enumerate(folds):
        fp = os.path.join(a.out, f"fold_{sid}.json")
        if os.path.exists(fp):
            done = json.load(open(fp))
            # --reeval redoes only folds evaluated by an older version (missing a metric), so it can resume
            if not a.reeval or all(x in fr for fr in done["fracs"].values() for x in EXTRA_METRICS):
                results.append(done)
                log(f"[{sid}] already done, loaded")
                continue
        torch.manual_seed(a.seed + fi)
        log("=" * 78)
        log(f"[{fi + 1}/{len(folds)}] TEST {sid}")
        pool = [s for s in subjects if s[0] != sid]
        order = np.random.default_rng(a.seed + fi).permutation(len(pool))
        val = [pool[i] for i in order[:a.n_val_subj]]
        train = [pool[i] for i in order[a.n_val_subj:]]
        log(f"  train={[s[0] for s in train]} val={[s[0] for s in val]}")
        t0 = time.time()
        enc_path = os.path.join(a.out, f"enc_{sid}.pt")
        if os.path.exists(enc_path):
            ck = torch.load(enc_path, map_location=dev, weights_only=False)
            enc = TemporalEncoder(**ck["cfg"]).to(dev)
            enc.load_state_dict(ck["state"])
            lam, best, hist = ck["lam"], ck["best_val"], ck["hist"]
            log(f"  loaded trained encoder (val r={best:.4f})")
        else:
            ckpt = os.path.join(a.out, f"ckpt_{sid}.pt")
            enc, lam, best, hist = train_encoder(a, train, val, dev, log, ckpt)
            torch.save({"state": enc.state_dict(), "cfg": enc.cfg, "lam": lam, "best_val": best, "hist": hist},
                       enc_path)
            if os.path.exists(ckpt):
                os.remove(ckpt)
        enc0 = TemporalEncoder(**enc.cfg).to(dev)
        log(f"  encoder: val r {hist[0][1]:.4f} (init) -> {best:.4f} (trained), lam={lam:.3g}, "
            f"{enc.n_feat} features/channel | {time.time() - t0:.0f}s")
        r = evaluate(E, Y, enc0, enc, a, dev, log)
        r.update(subject=sid, val_subjects=[s[0] for s in val], val_r_init=hist[0][1], val_r=best, lam=lam,
                 minutes=(time.time() - t0) / 60)
        with open(fp + ".tmp", "w") as fh:
            json.dump(r, fh)
        os.replace(fp + ".tmp", fp)
        results.append(r)
        log(f"[{sid}] done in {(time.time() - t0) / 60:.1f} min")
        del enc, enc0
        if dev.type == "cuda":
            torch.cuda.empty_cache()
    if results:
        summarize(results, a.out, log)
        key_tests(results, a.out, log)
        vi = [r["val_r_init"] for r in results]
        vt = [r["val_r"] for r in results]
        log(f"validation subjects, ridge readout on 20 %: Morlet init r={np.mean(vi):.4f} -> trained {np.mean(vt):.4f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/curve.py
"""
Phase 3a - calibration-size curve of the shared-encoder model (no retraining).

Uses the encoders saved by s2i.shared (enc_<sub>.pt, each trained WITHOUT that subject).
Fixed test set = last `test_frac` of trials after a 1-trial gap (as the Phase-0 curve);
calibration = the first f of trials. Models: ridge (train_loso baseline), base
(ridge_kfold_shared) and enc_ridge_cal (shared encoder + amplitude-calibrated ridge readout).
Question: how many minutes of calibration does the linear baseline need to match the encoder?

Usage
-----
python -m s2i.curve --data DATA_DIR --run RUNS_DIR/shared_bip_car
"""
import argparse
import glob
import json
import os
import sys
import time

import numpy as np
import torch

from s2i.ceiling import StreamRidge, load_trials
from s2i.metrics import per_contact_metrics
from s2i.shared import TemporalEncoder, amplitude_calibrate, np_feat
from s2i.within import ALPHAS_WIDE, LAG9, KFoldRidge

FS = 200
MODELS = ("ridge", "base", "enc_ridge_cal")


def analyze(E, Y, enc, a, dev, log):
    amp = dev.type == "cuda"
    n = len(E)
    t0 = int(round(n * (1 - a.test_frac)))
    test = list(range(min(n, t0 + 1), n))
    Et, Yt = [E[i] for i in test], [Y[i] for i in test]
    feat = np_feat(enc, dev, amp)
    res = {"n_trials": n, "n_test": len(test), "curve": {}}
    for f in a.fracs:
        n_c = int(round(n * f))
        if n_c < 3 or n_c > t0:
            continue
        Ec, Yc = E[:n_c], Y[:n_c]
        preds = {"ridge": StreamRidge(LAG9).fit(Ec, Yc).predict(Et),
                 "base": KFoldRidge(LAG9, shared=True, keep_oof=False).fit(Ec, Yc).predict(Et)}
        kf = KFoldRidge(feat, ALPHAS_WIDE, shared=True).fit(Ec, Yc)
        preds["enc_ridge_cal"] = amplitude_calibrate(kf, Yc, kf.predict(Et))
        row = {"n_calib": n_c, "minutes": sum(e.shape[1] for e in Ec) / FS / 60}
        for m, P in preds.items():
            mt = per_contact_metrics(P, Yt, FS, bands=False)
            row[m] = float(np.nanmean(mt["r"]))
            row[m + "_r2"] = float(np.nanmean(mt["r2"]))
        res["curve"][f"{f:g}"] = row
        log(f"    f={f:g} ({row['minutes']:.1f} min) " + " ".join(f"{m}={row[m]:.3f}" for m in MODELS))
    return res


def equivalent_minutes(xm, yb, target):
    """Minutes at which the (mean) baseline curve reaches `target`, by linear interpolation."""
    for i in range(1, len(xm)):
        if yb[i - 1] < target <= yb[i]:
            return xm[i - 1] + (target - yb[i - 1]) * (xm[i] - xm[i - 1]) / (yb[i] - yb[i - 1])
    return float("nan") if target > max(yb) else xm[0]


def summarize(results, out_dir, log):
    from scipy.stats import wilcoxon
    fracs = sorted({f for r in results for f in r["curve"]}, key=float)
    lines = ["| calib frac | n subj | minutes | " + " | ".join(MODELS) + " | enc − base | wins | Wilcoxon p | "
             "R² base | R² enc |", "|---" * (len(MODELS) + 8) + "|"]
    summ = {}
    for f in fracs:
        rows = [r["curve"][f] for r in results if f in r["curve"]]
        M = {m: np.array([r[m] for r in rows]) for m in MODELS}
        d = M["enc_ridge_cal"] - M["base"]
        try:
            p = float(wilcoxon(M["enc_ridge_cal"], M["base"]).pvalue)
        except ValueError:
            p = float("nan")
        summ[f] = {"n": len(rows), "minutes": float(np.mean([r["minutes"] for r in rows])),
                   **{m: float(M[m].mean()) for m in MODELS}, "diff": float(d.mean()), "wins": int((d > 0).sum()),
                   "p": p, "r2_base": float(np.mean([r["base_r2"] for r in rows])),
                   "r2_enc": float(np.mean([r["enc_ridge_cal_r2"] for r in rows]))}
        s = summ[f]
        lines.append(f"| {f} | {s['n']} | {s['minutes']:.1f} | " + " | ".join(f"{s[m]:.3f}" for m in MODELS) +
                     f" | {s['diff']:+.4f} | {s['wins']}/{s['n']} | {p:.4f} | {s['r2_base']:.3f} | {s['r2_enc']:.3f} |")

    # minutes-equivalence on subjects that have every fraction (consistent mean curves)
    full = [r for r in results if all(f in r["curve"] for f in fracs)]
    eq = []
    if len(full) >= 3:
        xm = [float(np.mean([r["curve"][f]["minutes"] for r in full])) for f in fracs]
        yb = [float(np.mean([r["curve"][f]["base"] for r in full])) for f in fracs]
        ye = [float(np.mean([r["curve"][f]["enc_ridge_cal"] for r in full])) for f in fracs]
        lines += ["", f"### minutes the baseline (`base`) needs to match the encoder ({len(full)} subjects with every "
                      "calibration size)",
                  "| encoder minutes | encoder r | base needs (min) | ratio |", "|---|---|---|---|"]
        for x, y in zip(xm, ye):
            need = equivalent_minutes(xm, yb, y)
            eq.append({"minutes": x, "enc_r": y, "base_minutes": need})
            lines.append(f"| {x:.1f} | {y:.3f} | " + (f"{need:.1f} | {need / x:.1f}x |" if np.isfinite(need)
                                                      else f"> {xm[-1]:.1f} (never reached) | - |"))
    md = "\n".join(lines)
    log("\n" + md)
    with open(os.path.join(out_dir, "curve.md"), "w", encoding="utf-8") as fh:
        fh.write(md + "\n")
    with open(os.path.join(out_dir, "curve.json"), "w") as fh:
        json.dump({"by_frac": summ, "equivalence": eq}, fh, indent=1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--run", required=True, help="output dir of s2i.shared (contains enc_<sub>.pt)")
    ap.add_argument("--fracs", type=float, nargs="*", default=[0.05, 0.1, 0.2, 0.3, 0.45, 0.59])
    ap.add_argument("--test_frac", type=float, default=0.4)
    ap.add_argument("--overwrite", action="store_true")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    out = os.path.join(a.run, "curve")
    os.makedirs(out, exist_ok=True)
    log_f = open(os.path.join(out, "curve.log"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} | {msg}"
        print(line, flush=True)
        log_f.write(line + "\n")
        log_f.flush()

    dev = torch.device(a.device)
    log(f"config: {vars(a)}")
    results = []
    for ep in sorted(glob.glob(os.path.join(a.run, "enc_sub-*.pt"))):
        sid = os.path.basename(ep)[4:-3]
        fp = os.path.join(out, f"curve_{sid}.json")
        if os.path.exists(fp) and not a.overwrite:
            results.append(json.load(open(fp)))
            log(f"[{sid}] already done, loaded")
            continue
        t0 = time.time()
        ck = torch.load(ep, map_location=dev, weights_only=False)
        enc = TemporalEncoder(**ck["cfg"]).to(dev)
        enc.load_state_dict(ck["state"])
        _, E, Y = load_trials(os.path.join(a.data, f"{sid}.pt"), "bip", "car")
        log(f"[{sid}] {len(E)} trials")
        r = analyze(E, Y, enc, a, dev, log)
        r["subject"] = sid
        with open(fp + ".tmp", "w") as fh:
            json.dump(r, fh)
        os.replace(fp + ".tmp", fp)
        results.append(r)
        log(f"[{sid}] done in {time.time() - t0:.0f}s")
    if results:
        summarize(results, out, log)


if __name__ == "__main__":
    main()

## 3A. Đường cong theo lượng calib (không train lại)
- **Tập test cố định:** 40% trial cuối, giống đường cong của giai đoạn 0.
- **Calib:** 5, 10, 20, 30, 45, 59% trial đầu.
- **Model:** `ridge`, `base`, `enc_ridge_cal`.
- Mỗi subject dùng encoder đã được train **không có** subject đó (`enc_<sub>.pt` của lần chạy chính).

Thời gian: khoảng 1 giờ. Chạy lại được nhiều lần.

In [ ]:
#@title 3A. Chạy đường cong
!python -m s2i.curve --data {DATA_DIR} --run {MAIN_RUN}

In [ ]:
#@title 3A. Bảng + hình
import json, numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display
CUR = f'{MAIN_RUN}/curve'
display(Markdown(open(f'{CUR}/curve.md', encoding='utf-8').read()))
R = [json.load(open(f)) for f in sorted(glob.glob(f'{CUR}/curve_sub-*.json'))]
fr = sorted({f for r in R for f in r['curve']}, key=float)
full = [r for r in R if all(f in r['curve'] for f in fr)]
print(f'{len(full)}/{len(R)} subject có đủ mọi mức calib (dùng cho hình)')
style = {'ridge': ('#9a9a9a', 'ridge (train_loso)'), 'base': ('#4c72b0', 'base (ridge K-fold)'),
         'enc_ridge_cal': ('#c44e52', 'encoder dùng chung + ridge')}
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
x = [np.mean([r['curve'][f]['minutes'] for r in full]) for f in fr]
for m, (c, lab) in style.items():
    for k, key in enumerate([m, m + '_r2']):
        v = np.array([[r['curve'][f][key] for f in fr] for r in full])
        mu, se = v.mean(0), v.std(0) / np.sqrt(len(v))
        ax[k].plot(x, mu, 'o-', color=c, label=lab); ax[k].fill_between(x, mu - se, mu + se, color=c, alpha=0.15)
ax[0].set_ylabel('mean r (test cố định)'); ax[1].set_ylabel('mean R²')
for a_ in ax:
    a_.set_xlabel('phút calib (trung bình theo subject)'); a_.grid(alpha=0.3)
ax[0].legend(); ax[0].set_title('Hiệu quả dữ liệu'); ax[1].set_title('Biên độ (R²)')
plt.tight_layout(); plt.savefig(f'{CUR}/curve.png', dpi=150); plt.show()

## 3B. Encoder giúp ở vùng nào? (calib 20%, từ kết quả của lần chạy chính)
r theo từng contact của `base` và `enc_ridge_cal`, gộp theo **độ sâu** và **vùng giải phẫu** (nhãn từ dữ liệu đã tiền xử lý).

Các contact trong cùng một subject không độc lập với nhau, nên bảng này chỉ để mô tả. Phép kiểm định chính vẫn là theo subject.

In [ ]:
#@title 3B. Theo độ sâu / vùng
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from s2i.data.dataset import Subject
rows = []
for fp in sorted(glob.glob(f'{MAIN_RUN}/fold_sub-*.json')):
    fo = json.load(open(fp)); sid = fo['subject']
    s = Subject(f'{DATA_DIR}/{sid}.pt', 'bip', 'car')
    rr = fo['fracs']['0.2']['r']
    for k in range(len(rr['base'])):
        rows.append({'subject': sid, 'contact': s.tgt_names[k], 'depth': s.tgt_depth[k], 'region': s.tgt_region[k],
                     'base': rr['base'][k], 'enc': rr['enc_ridge_cal'][k]})
    del s
df = pd.DataFrame(rows); df['diff'] = df['enc'] - df['base']
def table(col, min_n=1):
    g = df.groupby(col).agg(n=('diff', 'size'), n_subj=('subject', 'nunique'), base=('base', 'mean'),
                            enc=('enc', 'mean'), diff=('diff', 'mean'), improved=('diff', lambda d: (d > 0).mean()))
    return g[g.n >= min_n].sort_values('diff', ascending=False).round(3)
print(f'{len(df)} contact, {df.subject.nunique()} subject')
display(table('depth'))
display(table('region', min_n=15))
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].scatter(df.base, df.enc, s=6, alpha=0.4)
lo, hi = df[['base', 'enc']].min().min(), df[['base', 'enc']].max().max()
ax[0].plot([lo, hi], [lo, hi], 'k--', lw=0.8); ax[0].set_xlabel('base r'); ax[0].set_ylabel('encoder r')
ax[0].set_title(f'Theo contact: encoder tốt hơn ở {(df["diff"] > 0).mean():.0%} contact')
t = table('depth'); ax[1].bar(t.index.astype(str), t['diff']); ax[1].set_title('Mức tăng trung bình theo độ sâu')
ax[1].axhline(0, color='k', lw=0.8)
plt.tight_layout(); plt.savefig(f'{MAIN_RUN}/regions.png', dpi=150); plt.show()

## 3C. Độ ổn định theo seed và ablation (mỗi cấu hình khoảng 3 giờ GPU)
Chọn một cấu hình, chạy, rồi chọn cấu hình tiếp theo. Mỗi cấu hình lưu vào `runs/shared_<cấu hình>`, có resume, và dùng `--fast_eval`.

| Cấu hình | Câu hỏi | Ưu tiên |
|---|---|---|
| `seed1`, `seed2` | Kết quả có phụ thuộc vào seed và cách chọn subject validation không? | `seed1` đã xong (chênh lệch so với main = 0,0001). `seed2` không bắt buộc |
| `short` | Bộ lọc ±200 ms thay vì ±1,2 s: độ dài bộ lọc có quan trọng không? | 2 |
| `no_env` | Bỏ nhánh envelope (encoder hoàn toàn tuyến tính): phi tuyến có đóng góp không? | 2 |
| `long_train` | 6000 bước thay vì 3000 (val r chưa bão hoà ở 3000) | 3 |
| `wide` | 24 cặp + 8 bộ lọc tự do (gấp đôi đặc trưng) | 3 |

In [ ]:
#@title 3C. Chạy một cấu hình
PRESET = 'seed1'   #@param ["seed1", "seed2", "short", "no_env", "long_train", "wide"]
FLAGS = {'seed1': '--seed 1', 'seed2': '--seed 2', 'short': '--half 40', 'no_env': '--no_env',
         'long_train': '--steps 6000 --patience 10', 'wide': '--n_pairs 24 --n_free 8'}[PRESET]
OUT = f'{RUNS_DIR}/shared_{PRESET}'
print('->', OUT, '|', FLAGS)
!python -m s2i.shared --data {DATA_DIR} --out {OUT} --fast_eval {FLAGS}

## 3D. So sánh các lần chạy
So với lần chạy chính (`shared_bip_car`), trên các subject chung, theo cặp, ở calib 20%. Cấu hình nào chưa chạy đủ 15 fold vẫn được hiển thị với số fold hiện có.

In [ ]:
#@title 3D. Bảng so sánh
import json, numpy as np, pandas as pd
from scipy.stats import wilcoxon
def load(run, frac='0.2'):
    out = {}
    for fp in glob.glob(f'{run}/fold_sub-*.json'):
        fo = json.load(open(fp)); m = fo['fracs'][frac]['mean']
        out[fo['subject']] = {'base': m['base'], 'enc0': m.get('enc0_ridge', np.nan),
                              'enc': m.get('enc_ridge_cal', m.get('enc_ridge')), 'val_r': fo.get('val_r', np.nan)}
    return out
main = load(MAIN_RUN)
rows = []
for run in sorted(glob.glob(f'{RUNS_DIR}/shared_*')):
    d = load(run)
    common = sorted(set(d) & set(main))
    if not common:
        continue
    e = np.array([d[s]['enc'] for s in common]); b = np.array([d[s]['base'] for s in common])
    m0 = np.array([main[s]['enc'] for s in common]); e0 = np.array([d[s]['enc0'] for s in common])
    pw = lambda x, y: float(wilcoxon(x, y).pvalue) if len(x) >= 5 and np.any(x != y) else np.nan
    rows.append({'run': os.path.basename(run), 'folds': len(common), 'enc r': e.mean(), 'base r': b.mean(),
                 'enc − base': (e - b).mean(), 'wins vs base': f'{(e > b).sum()}/{len(e)}', 'p vs base': pw(e, b),
                 'enc − enc0': np.nanmean(e - e0), 'enc − main': (e - m0).mean(), 'p vs main': pw(e, m0),
                 'val r': np.nanmean([d[s]['val_r'] for s in common])})
display(pd.DataFrame(rows).set_index('run').round(4))

## 3E. So sánh với model deep end-to-end (LeadFormer, `train_loso.py`) trên cùng split
`train_loso.py` dùng đúng split này: 20% trial đầu để calib, gap 1 trial, phần còn lại để test. Cell này tự tìm mọi lần chạy `train_loso` (target bipolar, EEG CAR) trong `runs/`, ví dụ `runs/bip_car`.

**Kiểm tra trước:** cột `ridge (train_loso)` phải xấp xỉ `ridge (ours)`. Nếu lệch nhiều thì split hoặc dữ liệu đã khác.

In [ ]:
#@title 3E. Encoder dùng chung so với LeadFormer
import json, numpy as np, pandas as pd
from scipy.stats import wilcoxon
enc = {}
for fp in glob.glob(f'{MAIN_RUN}/fold_sub-*.json'):
    fo = json.load(open(fp)); m = fo['fracs']['0.2']['mean']
    enc[fo['subject']] = {'ridge (ours)': m['ridge'], 'base': m['base'], 'encoder': m['enc_ridge_cal']}
found = False
for run in sorted(glob.glob(f'{RUNS_DIR}/*')):
    cfgp = f'{run}/config.json'
    if not os.path.exists(cfgp):
        continue
    cfg = json.load(open(cfgp))
    if cfg.get('target') != 'bip' or cfg.get('eeg_ref') != 'car' or 'calib_frac' not in cfg:
        continue
    rows = []
    for fp in glob.glob(f'{run}/fold_sub-*.json'):
        fo = json.load(open(fp)); sid = fo['subject']
        if sid not in enc or 'calibrated' not in fo:
            continue
        rows.append({'subject': sid, 'LeadFormer zero-shot': np.nanmean(fo['zero_shot']['r']),
                     'LeadFormer calibrated': np.nanmean(fo['calibrated']['r']),
                     'ridge (train_loso)': np.nanmean(fo['ridge']['r']) if 'ridge' in fo else np.nan, **enc[sid]})
    if not rows:
        continue
    found = True
    df = pd.DataFrame(rows).set_index('subject').sort_index()
    print(f'== {os.path.basename(run)} (calib_frac={cfg["calib_frac"]}): {len(df)} subject chung')
    display(df.round(3)); display(df.mean().round(4).to_frame('mean r').T)
    if len(df) >= 5:
        d = df['encoder'] - df['LeadFormer calibrated']
        print(f"encoder − LeadFormer calibrated: {d.mean():+.4f}, thắng {(d > 0).sum()}/{len(df)}, "
              f"p={wilcoxon(df['encoder'], df['LeadFormer calibrated']).pvalue:.4f}")
if not found:
    print('Không tìm thấy lần chạy train_loso (bip + car) nào có fold_*.json trong', RUNS_DIR)

## 3F. Phần single-trial: encoder có dịch được tín hiệu theo từng trial, hay chỉ đoán trúng ERP?
Sternberg là task có cấu trúc trial cố định, nên cả EEG lẫn iEEG đều chứa thành phần khoá theo kích thích (ERP). Metric mới **`r_induced`** trừ khỏi **mỗi bên** trung bình theo trial của chính nó trên tập test, rồi mới tính r. Kiểm tra trên dữ liệu giả cho thấy:
- Dự đoán chỉ chứa ERP: r = 0,90 nhưng `r_induced` = 0.
- Không có ERP: `r_induced` = r.

Cell này đánh giá lại lần chạy chính bằng encoder đã lưu, không train lại, khoảng 1 giờ. Bảng **key comparisons** sẽ có thêm dòng `r_induced`.

In [ ]:
#@title 3F. Đánh giá lại lần chạy chính với r_induced
!python -m s2i.shared --data {DATA_DIR} --out {MAIN_RUN} --reeval

## 4. Cách đọc kết quả và hướng viết bài

**3A (hình chính).** "Baseline cần X phút để bằng encoder dùng Y phút." Nếu tỉ lệ X/Y ≥ 2 thì đây là thông điệp chính của bài: encoder học từ các bệnh nhân khác **giảm một nửa (hoặc hơn) thời gian ghi calib** cho bệnh nhân mới.

**3B.**
- Nếu mức tăng tập trung ở contact nông (vỏ não): phù hợp với dẫn truyền thể tích.
- Nếu có ở cả contact sâu (hồi hải mã, hạnh nhân): đây là điểm mạnh đáng nhấn mạnh trong bài.

**3C/3D.**
| Quan sát | Ý nghĩa |
|---|---|
| `seed1`, `seed2`: `enc − base` ≈ +0,027, 15/15 | Kết quả ổn định, báo cáo trung bình ± SD theo seed |
| `short` < main rõ | Bộ lọc **dài** là thành phần quan trọng, khớp với phát hiện của giai đoạn 1d |
| `no_env` ≈ main | Encoder tuyến tính là đủ, nên dùng model đơn giản hơn |
| `long_train` hoặc `wide` > main rõ | Còn dư địa. Cấu hình tốt nhất thành model cuối, sau đó chạy lại seed cho cấu hình đó |

Gửi lại: bảng và hình 3A, các bảng 3B, và bảng 3D sau mỗi cấu hình chạy xong.